In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import time 
import SimpleITK as sitk

In [ ]:
os.system("scripts/compile_project.sh 192 191 155 1.5 1.5 1.5 > compilation.log 2>&1")
with open("compilation.log") as f:
    print(f.read())

In [ ]:
# Add the relative path to the build directory
sys.path.insert(0, os.path.join(os.getcwd(), 'build'))

#import mlmcprotons
from mlmcprotons import MLMCprotons

In [ ]:
seed = 123456789
conversionTable = "HU2material.csv"
phantom = "CT.dat"
maxNumthreads = 8
miniStepShift = 1e-6        # Can not have integer step sizes

protonSim = MLMCprotons(seed, conversionTable)
protonSim.loadPhantom(phantom, 192, 191, 155)

# z = 79 * 1.5
# x = 100 * 1.5

protonSim.simulateBeam(0, 100000, 1.5+miniStepShift, 'y', 3, 0, 1, 0, 150, 0, 118, 160, 1.5, maxNumthreads, 0) 

dose = protonSim.yieldDose(0).grid
doseSlice = dose[:,:,79]

# Plotting the dose slice
plt.imshow(doseSlice, cmap='hot', interpolation='nearest')
plt.title('Dose Distribution')
plt.xlabel('Z [mm]')
plt.ylabel('Y [mm]')
plt.show()

In [ ]:

doseSlice = dose[:,:,79]

# Plotting the dose slice
plt.imshow(doseSlice, cmap='hot', interpolation='nearest')
plt.title('Dose Distribution')
plt.xlabel('Z [mm]')
plt.ylabel('Y [mm]')
plt.show()

In [ ]:
mydose = sitk.GetImageFromArray(dose)
mydose.SetSpacing((1.5, 1.5, 1.5))
mydose.SetOrigin((0, 0, 0))

sitk.WriteImage(mydose, "mydose.mha")

In [ ]:
os.chdir("CT_phantom")

CT_centered = sitk.ReadImage("CT.mhd")
CT_centered.SetOrigin((0.0, 0.0, 0.0))
sitk.WriteImage(CT_centered, "CT_centered.mha")